# Storage CSI DriversA practical refresher on the AWS storage **CSI (Container Storage Interface) drivers** used to backAI/ML workloads on Kubernetes (EKS): the **EBS**, **EFS**, **FSx for Lustre**, **FSx for OpenZFS**,and **Mountpoint for Amazon S3** CSI drivers.CSI is the standard plugin API that lets Kubernetes provision and attach storage from any vendorwithout that vendor's code living in the Kubernetes core. On AWS you install one driver per storageservice; each ships a `StorageClass` (or you author your own) and then `PersistentVolumeClaim`s(PVCs) bind to volumes that pods mount. The hard part of running training and inference at scale isalmost never compute — it's getting terabytes of data to the GPUs fast enough and sharing it acrossmany pods. Picking the right driver is how you solve that.

## Table of Contents1. [Introduction](#introduction)2. [Key Features](#key-features)3. [Architecture Overview](#architecture)4. [Installation](#installation)5. [Basic Usage](#basic-usage)6. [Advanced Features](#advanced-features)7. [Use Cases](#use-cases)8. [Best Practices](#best-practices)9. [Common Pitfalls](#pitfalls)10. [Performance Optimization](#performance)11. [Production Deployment](#deployment)12. [Monitoring and Observability](#monitoring)13. [Troubleshooting](#troubleshooting)14. [Comparison with Alternatives](#comparison)15. [Resources](#resources)

## Introduction <a id="introduction"></a>The Container Storage Interface (CSI) is a vendor-neutral API that lets Kubernetes provision, attach,mount, snapshot, and resize storage through out-of-tree driver pods instead of code baked into thekubelet. AWS maintains five CSI drivers that matter for AI/ML, each fronting a different storageservice. They are the bridge between a `PersistentVolumeClaim` in your manifest and a real volume inyour AWS account.### What is it?A CSI driver is a pair of components — a cluster-wide **controller** (a Deployment that talks to theAWS API to create/delete/attach volumes) and a per-node **node plugin** (a DaemonSet that mounts thevolume into the pod's filesystem namespace). The five AWS drivers are:| Driver | Backing service | Storage type | Access | Best for ||--------|-----------------|--------------|--------|----------|| `ebs.csi.aws.com` | Amazon EBS | Block (single-AZ) | ReadWriteOnce | Per-pod scratch, model weights, databases || `efs.csi.aws.com` | Amazon EFS | NFS (multi-AZ) | ReadWriteMany | Shared config/checkpoints across pods & AZs || `fsx.csi.aws.com` | FSx for Lustre | Parallel FS | ReadWriteMany | High-throughput training datasets fed to GPUs || `fsx.openzfs.csi.aws.com` | FSx for OpenZFS | NFS (snapshots) | ReadWriteMany | Shared POSIX data needing snapshots/clones || `s3.csi.aws.com` | Mountpoint for S3 | Object-as-file | ReadWriteMany (read-heavy) | Mounting existing S3 datasets directly |### Why use it?- **Decouple storage from compute.** GPU nodes come and go (Spot, autoscaling); your data and  checkpoints must outlive them. CSI volumes persist independently of the pod.- **Right-size cost vs throughput.** EBS gp3 for cheap per-pod disk, FSx for Lustre when you need  hundreds of GB/s to a fleet of GPUs, S3/Mountpoint when the dataset already lives in a bucket and  you want to avoid a copy step.- **Standard, portable manifests.** PVC/StorageClass YAML looks the same regardless of driver, so  you change a storage backend by changing a `storageClassName`, not your application.- **Dynamic provisioning, expansion, and snapshots** are handled by Kubernetes through the driver  rather than by hand in the AWS console.### When to use it?Reach for an AWS storage CSI driver when:- You run training or inference on **EKS** and need volumes that survive pod restarts.- Multiple pods (data loaders, distributed training replicas) must **share the same dataset**.- Your dataset is too large to bake into a container image or fit on a node's instance store.- You need **AZ-resilient** shared state (checkpoints, feature stores) — EBS is single-AZ, so this  pushes you toward EFS, FSx, or S3.

## Key Features <a id="key-features"></a>### Core capabilities by driver| Capability | EBS | EFS | FSx Lustre | FSx OpenZFS | Mountpoint S3 ||------------|-----|-----|------------|-------------|----------------|| Access mode | RWO | RWX | RWX | RWX | RWX (read-mostly) || Spans Availability Zones | No (1 AZ) | Yes | No (1 AZ) | No (1 AZ) | Yes (regional) || Dynamic provisioning | Yes | Yes (via access points) | Yes | Yes | No (static PV only) || Volume snapshots (CSI) | Yes | No | Yes (backups) | Yes (native ZFS) | No || Online expansion | Yes | N/A (elastic) | Yes (per data-repo) | Yes | N/A || Throughput ceiling | ~1–4 GB/s per vol | ~10+ GB/s elastic | 100s of GB/s | 10s of GB/s | S3 bandwidth || POSIX semantics | Full | Full (NFSv4) | Full | Full | Partial (no random writes) || Pay model | Provisioned GB+IOPS | Per-GB used | Per-GB provisioned | Per-GB provisioned | S3 storage + requests |### Why each matters for AI/ML- **EBS gp3** decouples IOPS/throughput from capacity, so a small model-weights volume can still get  3,000 IOPS / 125 MB/s baseline (up to 16,000 IOPS / 1,000 MB/s) without over-provisioning size.- **EFS** is the only fully-elastic, multi-AZ, RWX option with zero capacity planning — ideal for  checkpoints written by training pods scattered across AZs.- **FSx for Lustre** is the throughput king and can be **linked to an S3 bucket** as its data  repository, lazy-loading objects on first read and exporting results back — the canonical pattern  for feeding large datasets to many GPUs.- **FSx for OpenZFS** gives you instant, space-efficient snapshots and clones — handy for  reproducible dataset versioning and fast branch-and-experiment workflows.- **Mountpoint for S3** mounts a bucket as a filesystem with **no provisioning and no copy**, perfect  for sequential, read-heavy training over data that already lives in S3.

## Architecture Overview <a id="architecture"></a>Every AWS CSI driver follows the same two-plane shape defined by the CSI spec.```                         ┌──────────────────────────── EKS control plane ───────────────┐                         │  PersistentVolumeClaim ──► external-provisioner ──► Controller │                         │           │                                          │ (gRPC) │                         └───────────┼──────────────────────────────────────────┼────────┘                                     │ bind                                      │ AWS API                                     ▼                                           ▼                         ┌──── PersistentVolume ────┐                ┌──── AWS storage service ────┐                         │  spec.csi.driver: ...     │                │  EBS vol / EFS fs / FSx fs   │                         └───────────┬──────────────┘                └──────────────┬──────────────┘                                     │ scheduled to node                            │ attach/mount                                     ▼                                              ▼   GPU/CPU node:  kubelet ──► CSI node DaemonSet (NodeStageVolume / NodePublishVolume) ──► /var/lib/kubelet/pods/.../volume                                     │                                     ▼                                  Pod sees a mounted directory```### Components1. **CSI Controller (Deployment).** Runs the sidecars `external-provisioner` (creates the volume on a   PVC), `external-attacher` (issues ControllerPublish/attach), `external-resizer`,   `external-snapshotter`, and the AWS driver container that makes the actual AWS API calls. Usually   `replicas: 2` with leader election for HA.2. **CSI Node plugin (DaemonSet).** Runs on every node; the kubelet calls it via a Unix socket under   `/var/lib/kubelet/plugins/<driver>/csi.sock` to `NodeStageVolume` (format + mount to a global   staging path) and `NodePublishVolume` (bind-mount into the pod). For network filesystems (EFS,   FSx, S3) there's no attach step — the node plugin mounts directly.3. **IAM identity.** The driver's service account is bound to an IAM role via **IRSA** (IAM Roles for   Service Accounts) or **EKS Pod Identity**, granting `ec2:CreateVolume`, `elasticfilesystem:*`,   `fsx:*`, or `s3:*` as appropriate. No node-wide credentials needed.4. **Kubernetes storage objects.** `StorageClass` (provisioner + parameters), `PersistentVolume`   (the bound volume), `PersistentVolumeClaim` (the pod's request), and optional   `VolumeSnapshotClass`/`VolumeSnapshot` for point-in-time copies.

## Installation <a id="installation"></a>### Prerequisites- An **EKS cluster** (1.23+) and `kubectl` / `eksctl` / `helm` configured against it.- An **OIDC provider** associated with the cluster so IRSA works:  `eksctl utils associate-iam-oidc-provider --cluster <name> --approve`.- For EFS/FSx/S3: the relevant **security groups** (NFS 2049 for EFS, Lustre 988/1018-1023 for FSx)  and **subnets** must allow traffic from the node ENIs.- This notebook is **infrastructure-oriented**: the real work happens through `kubectl`, `eksctl`,  and `helm`, not Python. The Python cells below are self-contained decision/calculation helpers that  run anywhere; the storage operations are shown as shell snippets you run against your cluster.### Install the EBS CSI driver (managed add-on, recommended)```bash# Create the IAM role and install as an EKS managed add-oneksctl create iamserviceaccount \  --name ebs-csi-controller-sa --namespace kube-system \  --cluster my-cluster --role-name AmazonEKS_EBS_CSI_DriverRole \  --attach-policy-arn arn:aws:iam::aws:policy/service-role/AmazonEBSCSIDriverPolicy \  --approve --role-onlyeksctl create addon --name aws-ebs-csi-driver --cluster my-cluster \  --service-account-role-arn arn:aws:iam::<ACCOUNT>:role/AmazonEKS_EBS_CSI_DriverRole --force```### Install the EFS, FSx, and S3 CSI drivers (Helm)```bash# EFShelm repo add aws-efs-csi-driver https://kubernetes-sigs.github.io/aws-efs-csi-driver/helm upgrade --install aws-efs-csi-driver aws-efs-csi-driver/aws-efs-csi-driver \  -n kube-system --set controller.serviceAccount.annotations."eks\.amazonaws\.com/role-arn"=arn:aws:iam::<ACCOUNT>:role/EKS_EFS_CSI_Role# FSx for Lustrehelm repo add aws-fsx-csi-driver https://kubernetes-sigs.github.io/aws-fsx-csi-driver/helm upgrade --install aws-fsx-csi-driver aws-fsx-csi-driver/aws-fsx-csi-driver -n kube-system# Mountpoint for Amazon S3helm repo add aws-mountpoint-s3-csi-driver https://awslabs.github.io/mountpoint-s3-csi-driver/helm upgrade --install aws-mountpoint-s3-csi-driver \  aws-mountpoint-s3-csi-driver/aws-mountpoint-s3-csi-driver -n kube-system```### Verify the drivers are healthy```bashkubectl get pods -n kube-system | egrep 'ebs|efs|fsx|s3'-csikubectl get csidrivers# Expect: ebs.csi.aws.com, efs.csi.aws.com, fsx.csi.aws.com, s3.csi.aws.com```

In [ ]:
# This notebook drives Kubernetes/AWS, so the runnable cells are pure-Python
# decision and sizing helpers — no cluster or AWS credentials required.

from dataclasses import dataclass

@dataclass
class Driver:
    name: str
    provisioner: str
    access_modes: tuple
    multi_az: bool
    note: str

DRIVERS = [
    Driver("EBS",          "ebs.csi.aws.com",          ("RWO",),        False, "Cheap per-pod block disk"),
    Driver("EFS",          "efs.csi.aws.com",          ("RWX",),        True,  "Elastic shared NFS, multi-AZ"),
    Driver("FSx Lustre",   "fsx.csi.aws.com",          ("RWX",),        False, "Highest throughput, S3-linked"),
    Driver("FSx OpenZFS",  "fsx.openzfs.csi.aws.com",  ("RWX",),        False, "Shared NFS with snapshots/clones"),
    Driver("Mountpoint S3","s3.csi.aws.com",           ("RWX",),        True,  "Mount a bucket, read-heavy"),
]

header = f'{"Driver":<14}{"Provisioner":<28}{"Access":<8}{"Multi-AZ":<10}Note'
print(header)
for d in DRIVERS:
    access = "/".join(d.access_modes)
    print(f'{d.name:<14}{d.provisioner:<28}{access:<8}{str(d.multi_az):<10}{d.note}')

## Basic Usage <a id="basic-usage"></a>The pattern is always the same: define (or use) a `StorageClass`, create a PVC, mount it in a pod.### EBS: dynamic per-pod block volume (gp3)```yamlapiVersion: storage.k8s.io/v1kind: StorageClassmetadata:  name: ebs-gp3provisioner: ebs.csi.aws.comvolumeBindingMode: WaitForFirstConsumer   # provision in the pod's AZ — avoids cross-AZ scheduling failuresallowVolumeExpansion: trueparameters:  type: gp3  iops: "6000"  throughput: "500"           # MB/s, independent of size on gp3  encrypted: "true"---apiVersion: v1kind: PersistentVolumeClaimmetadata:  name: model-cachespec:  accessModes: ["ReadWriteOnce"]  storageClassName: ebs-gp3  resources:    requests:      storage: 200Gi```### EFS: shared RWX volume across many pods```yamlapiVersion: storage.k8s.io/v1kind: StorageClassmetadata:  name: efs-scprovisioner: efs.csi.aws.comparameters:  provisioningMode: efs-ap          # dynamically create an EFS access point per PVC  fileSystemId: fs-0123456789abcdef0  directoryPerms: "700"  gidRangeStart: "1000"  gidRangeEnd: "2000"---apiVersion: v1kind: PersistentVolumeClaimmetadata:  name: shared-checkpointsspec:  accessModes: ["ReadWriteMany"]  storageClassName: efs-sc  resources:    requests:      storage: 100Gi            # advisory only — EFS is elastic```### Mount the PVC in a training pod```yamlapiVersion: v1kind: Podmetadata:  name: trainerspec:  containers:    - name: trainer      image: 763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-training:2.3.0-gpu-py311      command: ["python", "train.py", "--data", "/data", "--ckpt", "/ckpt"]      resources:        limits:          nvidia.com/gpu: 8      volumeMounts:        - { name: dataset, mountPath: /data, readOnly: true }        - { name: ckpt,    mountPath: /ckpt }  volumes:    - name: dataset      persistentVolumeClaim: { claimName: shared-checkpoints }    - name: ckpt      persistentVolumeClaim: { claimName: model-cache }``````bashkubectl apply -f trainer.yamlkubectl get pvc                       # STATUS should become Boundkubectl exec trainer -- df -h /data /ckpt```

## Advanced Features <a id="advanced-features"></a>### FSx for Lustre linked to an S3 data repositoryThis is the flagship training pattern: a Lustre filesystem **lazy-loads** objects from an S3 bucketon first access and can **export** results back. GPUs read at hundreds of GB/s while the durable copystays in S3.```yamlapiVersion: storage.k8s.io/v1kind: StorageClassmetadata:  name: fsx-lustre-scprovisioner: fsx.csi.aws.comparameters:  subnetId: subnet-0abc...               # one subnet, single-AZ  securityGroupIds: sg-0abc...  deploymentType: PERSISTENT_2  perUnitStorageThroughput: "1000"       # MB/s per TiB (250/500/1000 for PERSISTENT_2)  autoImportPolicy: NEW_CHANGED          # auto-discover new/updated S3 objects  s3ImportPath: s3://my-datasets/imagenet/  s3ExportPath: s3://my-datasets/imagenet/results/mountOptions:  - flock``````bash# After the PVC binds, prime the metadata so first reads don't stall on import:kubectl exec trainer -- bash -c 'find /data -type f | head'      # triggers lazy load```### CSI volume snapshots (EBS / FSx)```yamlapiVersion: snapshot.storage.k8s.io/v1kind: VolumeSnapshotClassmetadata:  name: ebs-snapclassdriver: ebs.csi.aws.comdeletionPolicy: Delete---apiVersion: snapshot.storage.k8s.io/v1kind: VolumeSnapshotmetadata:  name: model-cache-snapspec:  volumeSnapshotClassName: ebs-snapclass  source:    persistentVolumeClaimName: model-cache```Restore by setting `spec.dataSource` on a new PVC to the `VolumeSnapshot`. FSx for OpenZFS exposesnative ZFS snapshots and copy-on-write **clones**, so you can branch a multi-TB dataset in secondsfor an experiment without duplicating bytes.### Mountpoint for S3 (static PV)S3 has no dynamic provisioning — you pre-create a PV that points at the bucket.```yamlapiVersion: v1kind: PersistentVolumemetadata:  name: s3-imagenet-pvspec:  capacity: { storage: 1200Gi }        # ignored by S3, required by the API  accessModes: ["ReadWriteMany"]  mountOptions:    - allow-delete    - region us-east-1    - prefix imagenet/  csi:    driver: s3.csi.aws.com    volumeHandle: s3-imagenet-pv        # must be unique cluster-wide    volumeAttributes: { bucketName: my-datasets }```

In [ ]:
# Sizing helper: FSx for Lustre throughput scales with provisioned capacity.
# Aggregate throughput (MB/s) = storage_TiB * perUnitStorageThroughput.

def fsx_lustre_throughput(storage_tib: float, per_unit_mb_s: int) -> dict:
    valid = {250, 500, 1000}  # PERSISTENT_2 tiers (MB/s per TiB)
    if per_unit_mb_s not in valid:
        raise ValueError(f"perUnitStorageThroughput must be one of {sorted(valid)}")
    if storage_tib < 1.2 or (storage_tib * 10) % 12 not in (0,):
        # PERSISTENT_2 is provisioned in 1.2 TiB increments
        pass
    agg = storage_tib * per_unit_mb_s
    return {
        "storage_TiB": storage_tib,
        "per_unit_MB_s_per_TiB": per_unit_mb_s,
        "aggregate_throughput_GB_s": round(agg / 1000, 2),
    }

for size in (1.2, 4.8, 12.0, 48.0):
    print(fsx_lustre_throughput(size, 1000))

## Use Cases <a id="use-cases"></a>### Use Case 1: Distributed training over a large image dataset- **Context:** 200-node PyTorch DDP job, 8 GPUs each, training over a 40 TB image corpus already in  S3. The bottleneck is feeding shuffled samples to 1,600 data loaders.- **Implementation:** FSx for Lustre (`PERSISTENT_2`, 1000 MB/s/TiB) linked to the S3 bucket with  `autoImportPolicy: NEW_CHANGED`. The PVC is `ReadWriteMany`, mounted read-only at `/data` in every  pod. Checkpoints go to a separate EFS PVC so any AZ can resume.- **Results:** Hundreds of GB/s aggregate read throughput, near-zero idle GPU time waiting on I/O,  and no upfront copy of 40 TB — objects stream in on first touch and stay cached in Lustre.### Use Case 2: Shared, AZ-resilient checkpoints- **Context:** Spot GPU nodes are reclaimed mid-training; the replacement may land in a different AZ.- **Implementation:** EFS PVC (`ReadWriteMany`) for `/ckpt`. Training writes checkpoints every N  steps; on restart the new pod mounts the same EFS access point regardless of AZ and resumes.- **Results:** Spot interruptions cost minutes, not a full restart, and there's no single-AZ EBS  volume stranding the job.### Use Case 3: Low-latency model weights for inference autoscaling- **Context:** An inference Deployment scales 5→200 replicas; each pod needs 30 GB of model weights  locally with fast cold-start.- **Implementation:** EBS gp3 per pod via `WaitForFirstConsumer` (provisions in the replica's AZ),  or a read-only EFS/Mountpoint-S3 mount if weights are shared and updated rarely. Pre-bake a  `VolumeSnapshot` of the weights volume so new PVCs restore from snapshot instead of re-downloading.- **Results:** Sub-minute warm-up because weights come off a local block device or a cached snapshot  rather than over the network on every scale-out.

## Best Practices <a id="best-practices"></a>1. **Use `volumeBindingMode: WaitForFirstConsumer` for EBS.** EBS volumes are AZ-local; binding   immediately can provision a volume in an AZ where the pod can't be scheduled. Late binding   provisions in the pod's chosen AZ.2. **Match the driver to the access pattern, not habit.** RWX shared reads → FSx Lustre or   Mountpoint S3; durable shared writes across AZs → EFS or FSx OpenZFS; fast per-pod scratch → EBS   gp3/io2. Don't force EBS into a job that needs many readers.3. **Grant IAM via IRSA or Pod Identity, scoped tightly.** Give the controller service account only   the actions it needs (`ec2:CreateVolume`/`fsx:CreateFileSystem`/`s3:GetObject`…), not broad   wildcards or node-instance-profile credentials.4. **Always encrypt at rest and set `reclaimPolicy: Retain` for stateful data.** A default `Delete`   policy will destroy a multi-TB volume the moment its PVC is removed.5. **Pre-create and pre-warm FSx-from-S3 filesystems** for predictable training starts; lazy import   on first read can otherwise stall the first epoch.6. **Enable `allowVolumeExpansion`** so you can grow EBS/FSx online with `kubectl edit pvc` instead of   migrating data to a bigger volume.7. **Pin driver versions and test upgrades.** CSI drivers run privileged DaemonSets; treat them like   any other critical infra component with staged rollout and a tested rollback.

## Common Pitfalls <a id="pitfalls"></a>1. **Expecting EBS to be ReadWriteMany.** EBS is block storage attachable to one node at a time   (RWO). Mounting it in replicas across nodes fails with `Multi-Attach error`. Use EFS/FSx/S3 for   shared access.2. **Cross-AZ scheduling deadlock.** An EBS PV in `us-east-1a` plus a pod forced onto a node in   `us-east-1b` leaves the pod `Pending` forever. Fix with `WaitForFirstConsumer` and/or topology   spread constraints.3. **Missing/over-broad IAM.** PVCs stuck `Pending` with `failed to provision volume ... AccessDenied`   almost always means the controller's IRSA role lacks the needed permission — or the OIDC provider   was never associated.4. **Security groups blocking the mount.** EFS needs inbound 2049 (NFS) and FSx Lustre needs   988/1018-1023 from the node security group; otherwise `NodePublishVolume` hangs and the pod   `ContainerCreating`s indefinitely.5. **Treating Mountpoint S3 like a POSIX filesystem.** No random writes, no rename, no append; writes   are sequential, full-object PUTs. Great for read-heavy training, wrong for a database or   checkpoint stream.6. **`reclaimPolicy: Delete` on precious data.** Deleting a PVC (or a whole namespace) silently   deletes the underlying EBS/FSx volume. Use `Retain` for anything you can't recreate.7. **Forgetting the snapshot CRDs.** Volume snapshots need the external-snapshotter controller and   the `snapshot.storage.k8s.io` CRDs installed cluster-wide — they are not part of the CSI driver   itself.

## Performance Optimization <a id="performance"></a>### Configuration tuning- **EBS gp3 vs io2 Block Express.** gp3 decouples IOPS/throughput from size (baseline 3,000 IOPS /  125 MB/s, up to 16,000 / 1,000). For latency-sensitive, IOPS-heavy workloads needing >16k IOPS,  use io2 Block Express (up to 256,000 IOPS, sub-ms). Set `iops`/`throughput` in the StorageClass.- **FSx for Lustre throughput tier.** `perUnitStorageThroughput` (250/500/1000 MB/s per TiB on  PERSISTENT_2) and total provisioned TiB set the aggregate ceiling — see the sizing helper above.  Scratch deployment types give the highest burst for transient jobs.- **EFS throughput mode.** Use **Elastic** throughput for spiky workloads (scales automatically) or  Provisioned when you need a guaranteed floor independent of stored size. General Purpose  performance mode for latency; Max I/O only when you have thousands of clients.- **Mountpoint S3 tuning.** Increase `--read-part-size`/prefetch and use many parallel readers; S3  throughput scales with request concurrency and is highest for large sequential reads.### Benchmark a mounted volume```bash# Sequential read throughput (run inside a pod with the PVC mounted at /data)fio --name=seqread --directory=/data --rw=read --bs=1M --size=10G \    --numjobs=8 --ioengine=libaio --direct=1 --group_reporting# Random read IOPS (block volumes)fio --name=randread --directory=/data --rw=randread --bs=4k --size=4G \    --numjobs=4 --iodepth=32 --ioengine=libaio --direct=1 --group_reporting```

In [ ]:
# Quick estimator: EBS gp3 monthly cost given provisioned size + extra IOPS/throughput.
# Prices are illustrative us-east-1 list rates (USD); always confirm current pricing.

def gp3_monthly_cost(size_gb: int, iops: int = 3000, throughput_mb_s: int = 125) -> dict:
    GB_RATE = 0.08            # $/GB-month
    IOPS_RATE = 0.005         # $/provisioned IOPS-month above the free 3000
    TPUT_RATE = 0.04          # $/MB-s-month above the free 125
    storage = size_gb * GB_RATE
    extra_iops = max(0, iops - 3000) * IOPS_RATE
    extra_tput = max(0, throughput_mb_s - 125) * TPUT_RATE
    return {
        "size_gb": size_gb,
        "iops": iops,
        "throughput_mb_s": throughput_mb_s,
        "monthly_usd": round(storage + extra_iops + extra_tput, 2),
    }

print(gp3_monthly_cost(200))                       # baseline 200 GB
print(gp3_monthly_cost(200, iops=6000, throughput_mb_s=500))  # tuned for training scratch

## Production Deployment <a id="deployment"></a>CSI drivers are cluster infrastructure, not application code — there is no Dockerfile *you* build.You deploy and operate the driver itself. Below are the production-grade patterns.### Pin and roll out the driver as an EKS add-on```bash# List available versions, then pin one and roll forward with PRESERVE conflict resolutionaws eks describe-addon-versions --addon-name aws-ebs-csi-driver \  --kubernetes-version 1.30 --query 'addons[0].addonVersions[*].addonVersion'aws eks update-addon --cluster-name my-cluster --addon-name aws-ebs-csi-driver \  --addon-version v1.35.0-eksbuild.1 --resolve-conflicts PRESERVE```### Harden the controller (HA + resources) via Helm values```yaml# values.yaml for a CSI driver Helm releasecontroller:  replicaCount: 2                       # leader-elected HA  resources:    requests: { cpu: 100m, memory: 128Mi }    limits:   { cpu: 500m, memory: 512Mi }  serviceAccount:    annotations:      eks.amazonaws.com/role-arn: arn:aws:iam::<ACCOUNT>:role/EKS_CSI_Role  tolerations:    - key: CriticalAddonsOnly      operator: Existsnode:  tolerateAllTaints: true               # node plugin must run on every node, incl. GPU/Spot```### Default StorageClass and per-tier classes```bash# Make gp3 the cluster default and remove the old gp2 defaultkubectl patch storageclass ebs-gp3 -p \  '{"metadata":{"annotations":{"storageclass.kubernetes.io/is-default-class":"true"}}}'kubectl patch storageclass gp2 -p \  '{"metadata":{"annotations":{"storageclass.kubernetes.io/is-default-class":"false"}}}'```Ship a small set of named classes (`ebs-gp3`, `efs-sc`, `fsx-lustre-sc`) so app teams pick a tier byname and never hand-craft volume parameters.

## Monitoring and Observability <a id="monitoring"></a>### Key metrics to track- **CloudWatch (storage service level):**  - EBS: `VolumeReadOps`/`VolumeWriteOps`, `VolumeQueueLength`, `BurstBalance`, throughput.  - EFS: `BurstCreditBalance`, `PercentIOLimit`, `ClientConnections`, `DataReadIOBytes`.  - FSx Lustre: `DataReadBytes`/`DataWriteBytes`, `FreeDataStorageCapacity`, `MetadataOperations`.  - S3/Mountpoint: bucket request metrics, 5xx/throttle (`503 SlowDown`) rates.- **Kubernetes level:** PVC phase (`Bound`/`Pending`), `kubelet_volume_stats_used_bytes` /  `kubelet_volume_stats_capacity_bytes` (fill %), `kubelet_volume_stats_inodes_used`, volume  attach/mount errors, and the CSI sidecar gRPC error counters scraped from the controller.- **Driver health:** controller/node DaemonSet ready counts, restart counts, leader-election churn.### Wiring it up```bash# Volume fill % surfaces via kubelet metrics — scrape with Prometheus and alert at >85%kubectl get --raw /api/v1/nodes/<node>/proxy/metrics | grep kubelet_volume_stats_used_bytes# Inspect events when a PVC misbehaveskubectl describe pvc shared-checkpointskubectl get events --field-selector involvedObject.kind=PersistentVolumeClaim```### Logging best practices- Scrape the CSI **controller** and **node** pod logs (`kubectl logs -n kube-system <csi-pod> -c  <driver-container>`) into your log stack; provisioning/mount failures are reported there.- Alert on PVCs stuck `Pending` > 5 min and on `BurstBalance`/`BurstCreditBalance` trending to zero  (you're about to get throttled).- Use structured CloudWatch alarms per filesystem so a noisy training job doesn't mask a real  capacity problem on another.

## Troubleshooting <a id="troubleshooting"></a>### Issue 1: PVC stuck in `Pending`**Symptoms:** `kubectl get pvc` shows `Pending`; `kubectl describe pvc` shows`failed to provision volume` or `waiting for first consumer`.**Cause:** Missing/incorrect IAM (IRSA role or OIDC provider), a `WaitForFirstConsumer` class with noschedulable pod yet, an invalid StorageClass parameter, or the subnet/security group referenced by anFSx class being wrong.**Solution:** `kubectl describe pvc <name>` and read the events; check the controller logs(`kubectl logs -n kube-system deploy/ebs-csi-controller -c ebs-plugin`); confirm the IRSA role hasthe right policy and the OIDC provider is associated.### Issue 2: Pod stuck `ContainerCreating` with mount/attach errors**Symptoms:** `Unable to attach or mount volumes`, `Multi-Attach error for volume`, or an NFS/Lustremount that hangs.**Cause:** EBS RWO volume being attached to a second node (Multi-Attach); a stale attachment from acrashed node; or security groups blocking NFS 2049 (EFS) / Lustre 988 & 1018-1023 (FSx).**Solution:** For Multi-Attach, ensure only one pod/node uses the RWO PVC (or switch to RWX storage);force-detach a stale `VolumeAttachment` if a node died. For NFS/Lustre hangs, fix the security-groupingress between node ENIs and the filesystem, and verify the mount targets exist in the node's AZ.### Issue 3: Mountpoint S3 write fails with `Operation not supported`**Symptoms:** Random writes, renames, or appends to an S3-mounted path return errors; only sequentialfull-file writes work.**Cause:** Mountpoint for S3 maps to object semantics — it does not support partial/random writes orrename. This is by design, not a bug.**Solution:** Use it read-only for training data, or write whole objects sequentially. Forcheckpoint/append workloads, use EFS or FSx instead.

## Comparison with Alternatives <a id="comparison"></a>### How the AWS storage CSI drivers compare| Dimension | EBS | EFS | FSx Lustre | FSx OpenZFS | Mountpoint S3 ||-----------|-----|-----|------------|-------------|----------------|| Access | RWO | RWX | RWX | RWX | RWX (read-heavy) || Multi-AZ | No | Yes | No | No | Yes (regional) || Peak throughput | ~1–4 GB/s/vol | 10s GB/s elastic | **100s GB/s** | 10s GB/s | S3 bandwidth || Latency | Lowest (block) | NFS network | Low (parallel FS) | NFS network | Higher (object) || Snapshots/clones | Snapshots | No | Backups | **Native ZFS** | No || Cost model | Provisioned GB+IOPS | Per-GB used | Per-GB provisioned | Per-GB provisioned | **Cheapest at rest** || Provisioning | Dynamic | Dynamic | Dynamic | Dynamic | Static only |### When to choose whichChoose **EBS** when one pod needs fast, cheap, low-latency block storage (databases, per-podweights/scratch). Choose **EFS** when many pods across AZs need elastic shared POSIX storage withzero capacity planning (checkpoints, shared config). Choose **FSx for Lustre** when you must feed alarge dataset to a fleet of GPUs at maximum throughput, ideally linked to S3. Choose **FSx forOpenZFS** when you want shared NFS plus instant snapshots/clones for dataset versioning. Choose**Mountpoint for S3** when the data already lives in a bucket and the access pattern is sequential,read-heavy, and you want to skip the copy and pay only S3 prices.### Versus non-AWS optionsAgainst self-managed alternatives — **Rook/Ceph** (run your own distributed storage),**Portworx**/**OpenEBS** (cloud-agnostic CSI layers), or **NFS server provisioner** — the AWS driverstrade portability for being fully managed: no storage cluster to babysit, native AWS durability andencryption, and tight S3 integration. Pick a portable layer only if you need to run the same stackacross clouds or on-prem.

## Resources <a id="resources"></a>### Official documentation- AWS EBS CSI driver: https://github.com/kubernetes-sigs/aws-ebs-csi-driver- AWS EFS CSI driver: https://github.com/kubernetes-sigs/aws-efs-csi-driver- AWS FSx for Lustre CSI driver: https://github.com/kubernetes-sigs/aws-fsx-csi-driver- AWS FSx for OpenZFS CSI driver: https://github.com/kubernetes-sigs/aws-fsx-openzfs-csi-driver- Mountpoint for Amazon S3 CSI driver: https://github.com/awslabs/mountpoint-s3-csi-driver- EKS storage docs: https://docs.aws.amazon.com/eks/latest/userguide/storage.html- CSI specification: https://github.com/container-storage-interface/spec/blob/master/spec.md### Tutorials and guides- EKS EBS CSI add-on walkthrough: https://docs.aws.amazon.com/eks/latest/userguide/ebs-csi.html- Using FSx for Lustre with EKS: https://docs.aws.amazon.com/eks/latest/userguide/fsx-csi.html- Kubernetes CSI developer/driver list: https://kubernetes-csi.github.io/docs/drivers.html### Community resources- AWS Containers Roadmap (storage issues/RFCs): https://github.com/aws/containers-roadmap/issues- Kubernetes Slack `#aws` / `#sig-storage`: https://kubernetes.slack.com- Stack Overflow tag: https://stackoverflow.com/questions/tagged/amazon-eks### Related technologies- Kubernetes Volume Snapshots & `external-snapshotter`- IAM Roles for Service Accounts (IRSA) and EKS Pod Identity- Karpenter / Cluster Autoscaler (node provisioning that interacts with AZ-local EBS volumes)